In [1]:
"""
═══════════════════════════════════════════════════════════════════════════════════
  07B_Seg_Benchmark.ipynb — Cell 1: SETUP & DATA
═══════════════════════════════════════════════════════════════════════════════════

  GOAL: Benchmark 3 segmentation models on IDENTICAL FUSeg splits.
        Pick the best → integrate into willie MINI/BASE/XL.

  MODELS TO BENCHMARK:
  ┌──────────────┬─────────────────────────────┬──────────────────────────┐
  │  Cell 2:     │  U-Net (EfficientNet-B3)    │  ✅ Already done: 84.59% │
  │  Cell 3:     │  MedSAM (medical SAM)       │  Fine-tune on FUSeg      │
  │  Cell 4:     │  SAM2 (Meta, 2024)          │  Fine-tune on FUSeg      │
  │  Cell 5:     │  Aggregate & Pick Winner    │                          │
  └──────────────┴─────────────────────────────┴──────────────────────────┘

  DATA: FUSeg dataset — same locked splits as all other experiments.
═══════════════════════════════════════════════════════════════════════════════════
"""

import os, sys, time, random, warnings, json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from PIL import Image
from sklearn.metrics import accuracy_score

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

class CFG:
    ROOT          = "."
    ARTIFACT_DIR  = os.path.join(ROOT, "artifacts", "willie_v2")
    BASELINES_DIR = os.path.join(ARTIFACT_DIR, "baselines")
    FIGURES_DIR   = os.path.join(ARTIFACT_DIR, "figures")
    SEG_BENCH_DIR = os.path.join(ARTIFACT_DIR, "seg_benchmark")

    IMG_SIZE      = 512
    BATCH_SIZE    = 4
    NUM_WORKERS   = 4
    MAX_EPOCHS    = 50
    PATIENCE      = 10
    LR            = 1e-4
    WEIGHT_DECAY  = 1e-4
    SEED          = 42
    DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(CFG.SEG_BENCH_DIR, exist_ok=True)

print("=" * 80)
print("  🔧 07B: SEGMENTATION BENCHMARK — SETUP")
print("=" * 80)
print(f"  Device:      {CFG.DEVICE}")
print(f"  Benchmark:   {CFG.SEG_BENCH_DIR}")

# ══════════════════════════════════════════════════════════════════════════════
# 2. REPRODUCIBILITY
# ══════════════════════════════════════════════════════════════════════════════

def seed_everything(seed=CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print(f"  Seeds:       {CFG.SEED}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. LOAD LOCKED SEGMENTATION SPLITS
# ══════════════════════════════════════════════════════════════════════════════

LOCKED_DIR = os.path.join(CFG.ROOT, "artifacts", "willie_LOCKED_INPUTS", "tables")
seg_train_path = os.path.join(LOCKED_DIR, "ws_seg_manifest_fuseg_train.csv")
seg_val_path   = os.path.join(LOCKED_DIR, "ws_seg_manifest_fuseg_val.csv")

assert os.path.exists(seg_train_path), f"❌ Not found: {seg_train_path}"
assert os.path.exists(seg_val_path),   f"❌ Not found: {seg_val_path}"

seg_train_df = pd.read_csv(seg_train_path)
seg_val_df   = pd.read_csv(seg_val_path)

# Auto-detect column names
seg_img_col = None
for col in ["image_path", "img_path", "img", "image"]:
    if col in seg_train_df.columns:
        seg_img_col = col
        break
if seg_img_col is None:
    seg_img_col = seg_train_df.columns[0]

seg_mask_col = None
for col in ["mask_path", "mask", "mask_file", "seg_path"]:
    if col in seg_train_df.columns:
        seg_mask_col = col
        break
if seg_mask_col is None:
    seg_mask_col = [c for c in seg_train_df.columns if "mask" in c.lower()][0]

print(f"\n📂 Segmentation Data (FUSeg)")
print("-" * 80)
print(f"  Columns:    {list(seg_train_df.columns)}")
print(f"  Image col:  {seg_img_col}")
print(f"  Mask col:   {seg_mask_col}")
print(f"  Train:      {len(seg_train_df)} images")
print(f"  Val:        {len(seg_val_df)} images")

# Verify a few images exist
sample_img = seg_train_df.iloc[0][seg_img_col]
sample_mask = seg_train_df.iloc[0][seg_mask_col]
print(f"\n  Sample image: {sample_img}")
print(f"  Sample mask:  {sample_mask}")
print(f"  Image exists: {os.path.exists(sample_img)}")
print(f"  Mask exists:  {os.path.exists(sample_mask)}")

# ══════════════════════════════════════════════════════════════════════════════
# 4. SHARED DATASET & EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class SegDataset(Dataset):
    """Shared segmentation dataset for all benchmarks."""
    def __init__(self, df, img_col, mask_col, img_size=512, augment=False):
        self.df = df.reset_index(drop=True)
        self.img_col = img_col
        self.mask_col = mask_col
        self.img_size = img_size
        self.augment = augment
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row[self.img_col]).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        mask = Image.open(row[self.mask_col]).convert("L")
        mask = mask.resize((self.img_size, self.img_size), Image.NEAREST)

        img_t = transforms.ToTensor()(img)
        mask_t = transforms.ToTensor()(mask)
        mask_t = (mask_t > 0.5).float()

        if self.augment:
            if random.random() > 0.5:
                img_t = torch.flip(img_t, [2])
                mask_t = torch.flip(mask_t, [2])
            if random.random() > 0.5:
                img_t = torch.flip(img_t, [1])
                mask_t = torch.flip(mask_t, [1])

        img_t = self.normalize(img_t)
        return img_t, mask_t.squeeze(0)


def compute_dice_scores(model, dataloader, device, is_sam=False):
    """Compute per-image Dice scores. Works for any model outputting logits."""
    model.eval()
    dice_scores = []
    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device)
            if is_sam:
                outputs = model(images)
            else:
                outputs = model(images)
            if outputs.dim() == 4 and outputs.shape[1] == 1:
                outputs = outputs.squeeze(1)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()
            for i in range(preds.size(0)):
                p, m = preds[i], masks[i]
                if m.sum() == 0 and p.sum() == 0:
                    dice_scores.append(1.0)
                elif m.sum() == 0 or p.sum() == 0:
                    dice_scores.append(0.0)
                else:
                    inter = (p * m).sum()
                    dice = (2. * inter) / (p.sum() + m.sum())
                    dice_scores.append(dice.item())
    return dice_scores


class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight

    def forward(self, logits, targets):
        if logits.dim() == 4 and logits.shape[1] == 1:
            logits = logits.squeeze(1)
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(1, 2))
        union = probs.sum(dim=(1, 2)) + targets.sum(dim=(1, 2))
        dice = (2. * intersection + 1e-7) / (union + 1e-7)
        dice_loss = 1 - dice.mean()
        return self.bce_weight * bce_loss + (1 - self.bce_weight) * dice_loss


# ══════════════════════════════════════════════════════════════════════════════
# 5. CHECKPOINT FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def save_seg_checkpoint(name, data):
    path = os.path.join(CFG.SEG_BENCH_DIR, f"{name}_results.pt")
    data["_timestamp"] = datetime.now().isoformat()
    torch.save(data, path)
    print(f"  💾 Saved: {path}")
    return path

def load_seg_checkpoint(name):
    path = os.path.join(CFG.SEG_BENCH_DIR, f"{name}_results.pt")
    if os.path.exists(path):
        data = torch.load(path, map_location="cpu", weights_only=False)
        print(f"  ♻️  Loaded: {path}")
        return data
    return None


# ══════════════════════════════════════════════════════════════════════════════
# 6. BUILD DATALOADERS
# ══════════════════════════════════════════════════════════════════════════════

train_ds = SegDataset(seg_train_df, seg_img_col, seg_mask_col,
                      img_size=CFG.IMG_SIZE, augment=True)
val_ds   = SegDataset(seg_val_df, seg_img_col, seg_mask_col,
                      img_size=CFG.IMG_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE,
                          shuffle=True, num_workers=CFG.NUM_WORKERS,
                          pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE,
                          shuffle=False, num_workers=CFG.NUM_WORKERS,
                          pin_memory=True)

print(f"\n  ✅ Dataloaders ready:")
print(f"     Train: {len(train_ds)} images, {len(train_loader)} batches")
print(f"     Val:   {len(val_ds)} images, {len(val_loader)} batches")

# ══════════════════════════════════════════════════════════════════════════════
# 7. LOAD U-NET RESULTS (already done in 07_Baselines)
# ══════════════════════════════════════════════════════════════════════════════

unet_path = os.path.join(CFG.BASELINES_DIR, "unet_results.pt")
if os.path.exists(unet_path):
    unet_results = torch.load(unet_path, map_location="cpu", weights_only=False)
    print(f"\n  ✅ U-Net results loaded from 07_Baselines:")
    print(f"     Mean Dice:   {unet_results['dice_mean']:.4f}")
    print(f"     Median Dice: {unet_results['dice_median']:.4f}")
else:
    print(f"\n  ⚠️  U-Net results not found — will retrain in Cell 2")
    unet_results = None

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*80}")
print(f"  ✅ CELL 1 COMPLETE — Seg Benchmark Infrastructure Ready")
print(f"{'='*80}")
print(f"""
  Data:  {len(train_ds)} train / {len(val_ds)} val (FUSeg, {CFG.IMG_SIZE}×{CFG.IMG_SIZE})
  Loss:  BCE + Dice (0.5/0.5)
  LR:    {CFG.LR}, AdamW, CosineAnnealing, patience={CFG.PATIENCE}

  Benchmark:
    Cell 2: U-Net (Eff-B3)  {'— ✅ ' + f"{unet_results['dice_mean']:.2%}" if unet_results else '— needs training'}
    Cell 3: MedSAM           — fine-tune on FUSeg
    Cell 4: SAM2             — fine-tune on FUSeg
    Cell 5: Aggregate        — pick winner for WILLIE

  → Paste output, then I'll give Cell 2.
""")

  🔧 07B: SEGMENTATION BENCHMARK — SETUP
  Device:      cuda
  Benchmark:   artifacts/willie_v2/seg_benchmark
  Seeds:       42

📂 Segmentation Data (FUSeg)
--------------------------------------------------------------------------------
  Columns:    ['split', 'img', 'mask', 'img_name', 'img_exists', 'mask_exists']
  Image col:  img
  Mask col:   mask
  Train:      610 images
  Val:        400 images

  Sample image: data/FUSeg/train/images/0012.png
  Sample mask:  data/FUSeg/train/labels/0012.png
  Image exists: True
  Mask exists:  True

  ✅ Dataloaders ready:
     Train: 610 images, 153 batches
     Val:   400 images, 100 batches

  ✅ U-Net results loaded from 07_Baselines:
     Mean Dice:   0.8459
     Median Dice: 0.9120

  ✅ CELL 1 COMPLETE — Seg Benchmark Infrastructure Ready

  Data:  610 train / 400 val (FUSeg, 512×512)
  Loss:  BCE + Dice (0.5/0.5)
  LR:    0.0001, AdamW, CosineAnnealing, patience=10

  Benchmark:
    Cell 2: U-Net (Eff-B3)  — ✅ 84.59%
    Cell 3: MedSAM     

In [4]:
"""
═══════════════════════════════════════════════════════════════════════════════════
  Cell 2: MedSAM — FINE-TUNE ON FUSeg
═══════════════════════════════════════════════════════════════════════════════════
  📄 Ma et al., Nature Communications (2024)
  🔗 https://github.com/bowang-lab/MedSAM
  
  Strategy: Freeze ViT-B encoder, fine-tune mask decoder + prompt encoder
  Prompt: bbox from ground truth masks (simulates RT-DETR at inference)
  ✅ CHECKPOINT SAFE: resumes from last completed epoch if interrupted
═══════════════════════════════════════════════════════════════════════════════════
"""

existing = load_seg_checkpoint("medsam")
if existing is not None:
    print(f"  ⏩ SKIPPING MedSAM — final checkpoint found!")
    print(f"     Mean Dice: {existing['dice_mean']:.4f}")
    medsam_results = existing
else:
    print(f"\n{'='*80}")
    print(f"  🏋️  TRAINING: MedSAM on FUSeg")
    print(f"  📄 Ma et al., Nature Communications (2024)")
    print(f"{'='*80}")

    try:
        from segment_anything import sam_model_registry
        print("  ✅ segment_anything already installed")
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install",
                        "git+https://github.com/facebookresearch/segment-anything.git",
                        "--quiet"], check=True)
        from segment_anything import sam_model_registry
        print("  ✅ segment_anything installed")

    # ── Download weights ──
    medsam_ckpt_dir = os.path.join(CFG.ROOT, "pretrained_weights")
    os.makedirs(medsam_ckpt_dir, exist_ok=True)
    medsam_ckpt_path = os.path.join(medsam_ckpt_dir, "medsam_vit_b.pth")

    if not os.path.exists(medsam_ckpt_path):
        print("  📥 Downloading MedSAM weights...")
        import urllib.request
        try:
            medsam_url = "https://huggingface.co/wanglab/medsam/resolve/main/medsam_vit_b.pth"
            urllib.request.urlretrieve(medsam_url, medsam_ckpt_path)
            print(f"  ✅ MedSAM weights downloaded")
        except:
            print("  ⚠️  MedSAM weights unavailable, using SAM ViT-B base")
            url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
            urllib.request.urlretrieve(url, medsam_ckpt_path)
            print(f"  ✅ SAM ViT-B weights downloaded")
    else:
        print(f"  ✅ Weights found: {medsam_ckpt_path}")

    # ── Load model ──
    sam_model = sam_model_registry["vit_b"](checkpoint=medsam_ckpt_path)
    sam_model = sam_model.to(CFG.DEVICE)

    total_params = sum(p.numel() for p in sam_model.parameters())
    print(f"  MedSAM (ViT-B): {total_params/1e6:.1f}M total")

    for param in sam_model.image_encoder.parameters():
        param.requires_grad = False

    trainable = sum(p.numel() for p in sam_model.parameters() if p.requires_grad)
    print(f"  Trainable (decoder+prompt): {trainable/1e6:.2f}M")

    # ── Dataset with bbox prompts ──
    class MedSAMDataset(Dataset):
        def __init__(self, df, img_col, mask_col, img_size=1024, augment=False):
            self.df = df.reset_index(drop=True)
            self.img_col = img_col
            self.mask_col = mask_col
            self.img_size = img_size
            self.augment = augment

        def __len__(self):
            return len(self.df)

        def _get_bbox_from_mask(self, mask_np):
            ys, xs = np.where(mask_np > 0)
            if len(xs) == 0:
                return np.array([0, 0, self.img_size, self.img_size], dtype=np.float32)
            x1, x2 = xs.min(), xs.max()
            y1, y2 = ys.min(), ys.max()
            w, h = x2 - x1, y2 - y1
            pad_x = max(int(w * 0.05), 2)
            pad_y = max(int(h * 0.05), 2)
            x1 = max(0, x1 - pad_x)
            y1 = max(0, y1 - pad_y)
            x2 = min(self.img_size - 1, x2 + pad_x)
            y2 = min(self.img_size - 1, y2 + pad_y)
            return np.array([x1, y1, x2, y2], dtype=np.float32)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img = Image.open(row[self.img_col]).convert("RGB")
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
            mask = Image.open(row[self.mask_col]).convert("L")
            mask = mask.resize((self.img_size, self.img_size), Image.NEAREST)

            img_np = np.array(img, dtype=np.float32)
            mask_np = (np.array(mask) > 127).astype(np.float32)

            if self.augment:
                if random.random() > 0.5:
                    img_np = img_np[:, ::-1, :].copy()
                    mask_np = mask_np[:, ::-1].copy()
                if random.random() > 0.5:
                    img_np = img_np[::-1, :, :].copy()
                    mask_np = mask_np[::-1, :].copy()

            bbox = self._get_bbox_from_mask(mask_np)
            img_t = torch.from_numpy(img_np).permute(2, 0, 1).float()
            mask_t = torch.from_numpy(mask_np).float()
            bbox_t = torch.from_numpy(bbox).float()
            return img_t, mask_t, bbox_t

    # ── Dataloaders ──
    sam_train_ds = MedSAMDataset(seg_train_df, seg_img_col, seg_mask_col,
                                  img_size=1024, augment=True)
    sam_val_ds   = MedSAMDataset(seg_val_df, seg_img_col, seg_mask_col,
                                  img_size=1024, augment=False)
    sam_train_loader = DataLoader(sam_train_ds, batch_size=2, shuffle=True,
                                   num_workers=CFG.NUM_WORKERS, pin_memory=True)
    sam_val_loader   = DataLoader(sam_val_ds, batch_size=2, shuffle=False,
                                   num_workers=CFG.NUM_WORKERS, pin_memory=True)

    print(f"  Train: {len(sam_train_ds)}, Val: {len(sam_val_ds)}")
    print(f"  Image size: 1024×1024, Batch: 2")

    # ── Helper: per-image SAM forward ──
    def sam_forward_single(sam, emb, box):
        sparse_emb, dense_emb = sam.prompt_encoder(
            points=None, boxes=box, masks=None,
        )
        low_res, _ = sam.mask_decoder(
            image_embeddings=emb,
            image_pe=sam.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_emb,
            dense_prompt_embeddings=dense_emb,
            multimask_output=False,
        )
        pred = F.interpolate(low_res, (1024, 1024),
                             mode="bilinear", align_corners=False)
        return pred

    # ── Optimizer & Scheduler ──
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, sam_model.parameters()),
        lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.MAX_EPOCHS)
    criterion = BCEDiceLoss(bce_weight=0.5)

    best_dice = 0.0
    best_state = None
    patience_counter = 0
    start_epoch = 0
    start_time = time.time()

    # ══════════════════════════════════════════════════════════════════════
    # CHECKPOINT RESUME: load mid-training checkpoint if exists
    # ══════════════════════════════════════════════════════════════════════
    resume_path = os.path.join(CFG.SEG_BENCH_DIR, "medsam_training_ckpt.pt")
    if os.path.exists(resume_path):
        ckpt = torch.load(resume_path, map_location=CFG.DEVICE, weights_only=False)
        # Restore decoder/prompt weights
        current_sd = sam_model.state_dict()
        current_sd.update(ckpt["model_decoder_state"])
        sam_model.load_state_dict(current_sd)
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_dice = ckpt["best_dice"]
        best_state = ckpt["best_state"]
        patience_counter = ckpt["patience_counter"]
        print(f"  ♻️  RESUMING from epoch {start_epoch}, best_dice={best_dice:.4f}")

    # ── Training loop ──
    for epoch in range(start_epoch, CFG.MAX_EPOCHS):
        sam_model.train()
        sam_model.image_encoder.eval()
        train_loss = 0.0

        for images, masks, bboxes in sam_train_loader:
            images = images.to(CFG.DEVICE)
            masks  = masks.to(CFG.DEVICE)
            bboxes = bboxes.to(CFG.DEVICE)

            optimizer.zero_grad()

            with torch.no_grad():
                image_embeddings = sam_model.image_encoder(
                    sam_model.preprocess(images)
                )

            batch_loss = 0.0
            for b_idx in range(images.size(0)):
                emb = image_embeddings[b_idx].unsqueeze(0)
                box = bboxes[b_idx].unsqueeze(0)
                gt  = masks[b_idx].unsqueeze(0)

                pred = sam_forward_single(sam_model, emb, box)
                pred = pred.squeeze(1)
                batch_loss += criterion(pred, gt)

            loss = batch_loss / images.size(0)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)

        train_loss /= len(sam_train_ds)
        scheduler.step()

        # ── Validate ──
        sam_model.eval()
        val_dices = []
        with torch.no_grad():
            for images, masks, bboxes in sam_val_loader:
                images = images.to(CFG.DEVICE)
                masks  = masks.to(CFG.DEVICE)
                bboxes = bboxes.to(CFG.DEVICE)

                image_embeddings = sam_model.image_encoder(
                    sam_model.preprocess(images)
                )
                for b_idx in range(images.size(0)):
                    emb = image_embeddings[b_idx].unsqueeze(0)
                    box = bboxes[b_idx].unsqueeze(0)
                    gt  = masks[b_idx]

                    pred = sam_forward_single(sam_model, emb, box)
                    prob = torch.sigmoid(pred.squeeze(0).squeeze(0))
                    p = (prob > 0.5).float()
                    m = gt
                    if m.sum() == 0 and p.sum() == 0:
                        val_dices.append(1.0)
                    elif m.sum() == 0 or p.sum() == 0:
                        val_dices.append(0.0)
                    else:
                        inter = (p * m).sum()
                        dice = (2. * inter) / (p.sum() + m.sum())
                        val_dices.append(dice.item())

        mean_dice = np.mean(val_dices)
        median_dice = np.median(val_dices)

        if (epoch + 1) in [1, 5, 10, 15, 20, 30, 40, 50]:
            print(f"    Ep {epoch+1:3d}: loss={train_loss:.4f} "
                  f"mean_dice={mean_dice:.4f} median={median_dice:.4f}")

        if mean_dice > best_dice:
            best_dice = mean_dice
            best_state = {k: v.cpu().clone() for k, v in sam_model.state_dict().items()
                          if "mask_decoder" in k or "prompt_encoder" in k}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= CFG.PATIENCE:
                print(f"    ⏹ Early stop at epoch {epoch + 1}")
                break

        # ══════════════════════════════════════════════════════════════════
        # SAVE MID-TRAINING CHECKPOINT every 5 epochs
        # ══════════════════════════════════════════════════════════════════
        if (epoch + 1) % 5 == 0:
            decoder_state = {k: v.cpu().clone() for k, v in sam_model.state_dict().items()
                             if "mask_decoder" in k or "prompt_encoder" in k}
            torch.save({
                "epoch": epoch,
                "model_decoder_state": decoder_state,
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "best_dice": best_dice,
                "best_state": best_state,
                "patience_counter": patience_counter,
            }, resume_path)
            print(f"    💾 Mid-training checkpoint saved (epoch {epoch+1})")

    total_time = (time.time() - start_time) / 60

    # ── Final eval with best weights ──
    current_state = sam_model.state_dict()
    current_state.update(best_state)
    sam_model.load_state_dict(current_state)
    sam_model.eval()

    final_dices = []
    with torch.no_grad():
        for images, masks, bboxes in sam_val_loader:
            images = images.to(CFG.DEVICE)
            masks  = masks.to(CFG.DEVICE)
            bboxes = bboxes.to(CFG.DEVICE)

            image_embeddings = sam_model.image_encoder(
                sam_model.preprocess(images)
            )
            for b_idx in range(images.size(0)):
                emb = image_embeddings[b_idx].unsqueeze(0)
                box = bboxes[b_idx].unsqueeze(0)
                gt  = masks[b_idx]

                pred = sam_forward_single(sam_model, emb, box)
                prob = torch.sigmoid(pred.squeeze(0).squeeze(0))
                p = (prob > 0.5).float()
                m = gt
                if m.sum() == 0 and p.sum() == 0:
                    final_dices.append(1.0)
                elif m.sum() == 0 or p.sum() == 0:
                    final_dices.append(0.0)
                else:
                    inter = (p * m).sum()
                    dice = (2. * inter) / (p.sum() + m.sum())
                    final_dices.append(dice.item())

    dice_mean = np.mean(final_dices)
    dice_median = np.median(final_dices)
    dice_std = np.std(final_dices)

    print(f"\n  {'─'*60}")
    print(f"  📊 MedSAM — FINAL RESULTS")
    print(f"  {'─'*60}")
    print(f"  Mean Dice:    {dice_mean:.4f} ± {dice_std:.4f}")
    print(f"  Median Dice:  {dice_median:.4f}")
    print(f"  Total time:   {total_time:.1f} min")
    print(f"  Params:       {total_params/1e6:.1f}M total, {trainable/1e6:.2f}M trainable")
    print(f"\n  Comparison:")
    print(f"    U-Net (Eff-B3): {unet_results['dice_mean']:.4f} mean / {unet_results['dice_median']:.4f} median")

    medsam_results = {
        "dice_mean": dice_mean,
        "dice_median": dice_median,
        "dice_std": dice_std,
        "dice_scores": final_dices,
        "total_time_min": total_time,
        "total_params": total_params,
        "trainable_params": trainable,
    }

    save_seg_checkpoint("medsam", medsam_results)

    decoder_path = os.path.join(CFG.SEG_BENCH_DIR, "medsam_decoder_best.pt")
    torch.save(best_state, decoder_path)
    print(f"  💾 Decoder weights: {decoder_path}")

    # Clean up mid-training checkpoint (training complete)
    if os.path.exists(resume_path):
        os.remove(resume_path)
        print(f"  🧹 Cleaned mid-training checkpoint")

    del sam_model, optimizer, scheduler
    torch.cuda.empty_cache()

print(f"\n✅ Cell 2 COMPLETE — MedSAM Mean Dice: {medsam_results['dice_mean']:.4f}")
print(f"✅ Ready for Cell 3: SAM2")


  🏋️  TRAINING: MedSAM on FUSeg
  📄 Ma et al., Nature Communications (2024)
  ✅ segment_anything already installed
  ✅ Weights found: pretrained_weights/medsam_vit_b.pth
  MedSAM (ViT-B): 93.7M total
  Trainable (decoder+prompt): 4.06M
  Train: 610, Val: 400
  Image size: 1024×1024, Batch: 2
    Ep   1: loss=0.0770 mean_dice=0.8589 median=0.9203
    Ep   5: loss=0.0576 mean_dice=0.9093 median=0.9333
    💾 Mid-training checkpoint saved (epoch 5)
    Ep  10: loss=0.0560 mean_dice=0.8867 median=0.9312
    💾 Mid-training checkpoint saved (epoch 10)
    Ep  15: loss=0.0507 mean_dice=0.9093 median=0.9343
    ⏹ Early stop at epoch 15

  ────────────────────────────────────────────────────────────
  📊 MedSAM — FINAL RESULTS
  ────────────────────────────────────────────────────────────
  Mean Dice:    0.9093 ± 0.0803
  Median Dice:  0.9333
  Total time:   36.6 min
  Params:       93.7M total, 4.06M trainable

  Comparison:
    U-Net (Eff-B3): 0.8459 mean / 0.9120 median
  💾 Saved: artifacts/w

In [6]:
"""
═══════════════════════════════════════════════════════════════════════════════════
  Cell 3: SAM2 — FINE-TUNE ON FUSeg
═══════════════════════════════════════════════════════════════════════════════════
  📄 Ravi et al., Meta FAIR (2024)
  🔗 https://github.com/facebookresearch/sam2
  
  Strategy: Use SAM2ImagePredictor for encoding, fine-tune mask decoder
  ✅ CHECKPOINT SAFE
═══════════════════════════════════════════════════════════════════════════════════
"""

existing = load_seg_checkpoint("sam2")
if existing is not None:
    print(f"  ⏩ SKIPPING SAM2 — final checkpoint found!")
    print(f"     Mean Dice: {existing['dice_mean']:.4f}")
    sam2_results = existing
else:
    print(f"\n{'='*80}")
    print(f"  🏋️  TRAINING: SAM2 on FUSeg")
    print(f"  📄 Ravi et al., Meta FAIR (2024)")
    print(f"{'='*80}")

    try:
        from sam2.build_sam import build_sam2
        from sam2.sam2_image_predictor import SAM2ImagePredictor
        print("  ✅ sam2 already installed")
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install",
                        "sam2", "--quiet"], check=True)
        from sam2.build_sam import build_sam2
        from sam2.sam2_image_predictor import SAM2ImagePredictor
        print("  ✅ sam2 installed")

    # ── Weights ──
    sam2_ckpt_dir = os.path.join(CFG.ROOT, "pretrained_weights")
    os.makedirs(sam2_ckpt_dir, exist_ok=True)
    sam2_ckpt_path = os.path.join(sam2_ckpt_dir, "sam2.1_hiera_small.pt")
    sam2_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"

    if not os.path.exists(sam2_ckpt_path):
        print("  📥 Downloading SAM2.1 Hiera-Small weights...")
        import urllib.request
        url = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt"
        urllib.request.urlretrieve(url, sam2_ckpt_path)
        print(f"  ✅ Downloaded")
    else:
        print(f"  ✅ Weights found: {sam2_ckpt_path}")

    # ── Load model via predictor ──
    sam2_model = build_sam2(sam2_cfg, sam2_ckpt_path, device=str(CFG.DEVICE))
    predictor = SAM2ImagePredictor(sam2_model)

    total_params = sum(p.numel() for p in sam2_model.parameters())
    print(f"  SAM2.1 (Hiera-S): {total_params/1e6:.1f}M total")

    # Freeze image encoder
    for param in sam2_model.image_encoder.parameters():
        param.requires_grad = False

    for param in sam2_model.sam_mask_decoder.parameters():
        param.requires_grad = True
    for param in sam2_model.sam_prompt_encoder.parameters():
        param.requires_grad = True

    trainable = sum(p.numel() for p in sam2_model.parameters() if p.requires_grad)
    print(f"  Trainable (decoder+prompt): {trainable/1e6:.2f}M")

    # ── Dataset ──
    class SAM2Dataset(Dataset):
        def __init__(self, df, img_col, mask_col, img_size=1024, augment=False):
            self.df = df.reset_index(drop=True)
            self.img_col = img_col
            self.mask_col = mask_col
            self.img_size = img_size
            self.augment = augment

        def __len__(self):
            return len(self.df)

        def _get_bbox_from_mask(self, mask_np):
            ys, xs = np.where(mask_np > 0)
            if len(xs) == 0:
                return np.array([0, 0, self.img_size, self.img_size], dtype=np.float32)
            x1, x2 = xs.min(), xs.max()
            y1, y2 = ys.min(), ys.max()
            w, h = x2 - x1, y2 - y1
            pad_x = max(int(w * 0.05), 2)
            pad_y = max(int(h * 0.05), 2)
            x1 = max(0, x1 - pad_x)
            y1 = max(0, y1 - pad_y)
            x2 = min(self.img_size - 1, x2 + pad_x)
            y2 = min(self.img_size - 1, y2 + pad_y)
            return np.array([x1, y1, x2, y2], dtype=np.float32)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img = Image.open(row[self.img_col]).convert("RGB")
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
            mask = Image.open(row[self.mask_col]).convert("L")
            mask = mask.resize((self.img_size, self.img_size), Image.NEAREST)

            img_np = np.array(img, dtype=np.uint8)  # SAM2 expects uint8
            mask_np = (np.array(mask) > 127).astype(np.float32)

            if self.augment:
                if random.random() > 0.5:
                    img_np = img_np[:, ::-1, :].copy()
                    mask_np = mask_np[:, ::-1].copy()
                if random.random() > 0.5:
                    img_np = img_np[::-1, :, :].copy()
                    mask_np = mask_np[::-1, :].copy()

            bbox = self._get_bbox_from_mask(mask_np)
            mask_t = torch.from_numpy(mask_np).float()
            bbox_t = torch.from_numpy(bbox).float()
            return img_np, mask_t, bbox_t

    # ── Collate fn for numpy images ──
    def sam2_collate(batch):
        imgs, masks, bboxes = zip(*batch)
        masks = torch.stack(masks)
        bboxes = torch.stack(bboxes)
        return list(imgs), masks, bboxes

    s2_train_ds = SAM2Dataset(seg_train_df, seg_img_col, seg_mask_col,
                               img_size=1024, augment=True)
    s2_val_ds   = SAM2Dataset(seg_val_df, seg_img_col, seg_mask_col,
                               img_size=1024, augment=False)
    s2_train_loader = DataLoader(s2_train_ds, batch_size=1, shuffle=True,
                                  num_workers=0, collate_fn=sam2_collate)
    s2_val_loader   = DataLoader(s2_val_ds, batch_size=1, shuffle=False,
                                  num_workers=0, collate_fn=sam2_collate)

    print(f"  Train: {len(s2_train_ds)}, Val: {len(s2_val_ds)}")
    print(f"  Image size: 1024×1024, Batch: 1 (per-image processing)")

    # ── SAM2 forward using predictor internals ──
    @torch.no_grad()
    def encode_image(pred, img_np):
        """Encode image using SAM2ImagePredictor (handles all preprocessing)."""
        pred.set_image(img_np)
        return pred._features  # dict with image embeddings + high-res feats

    def decode_with_box(model, features, bbox, original_size=(1024, 1024)):
        """Decode mask from cached features + bbox prompt."""
        # bbox: (4,) tensor [x1, y1, x2, y2]
        box_input = bbox.unsqueeze(0).unsqueeze(0)  # (1, 1, 4)

        sparse_emb, dense_emb = model.sam_prompt_encoder(
            points=None, boxes=box_input, masks=None,
        )

        high_res_feats = [
            feat_level[-1].unsqueeze(0)
            for feat_level in features["high_res_feats"]
        ]
        image_embed = features["image_embed"][-1].unsqueeze(0)

        low_res_masks, _, _, _ = model.sam_mask_decoder(
            image_embeddings=image_embed,
            image_pe=model.sam_prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_emb,
            dense_prompt_embeddings=dense_emb,
            multimask_output=False,
            repeat_image=False,
            high_res_features=high_res_feats,
        )

        pred_masks = F.interpolate(
            low_res_masks.float(),
            original_size,
            mode="bilinear",
            align_corners=False,
        )
        return pred_masks.squeeze(1)  # (1, H, W)

    # ── Optimizer ──
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, sam2_model.parameters()),
        lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.MAX_EPOCHS)
    criterion = BCEDiceLoss(bce_weight=0.5)

    best_dice = 0.0
    best_state = None
    patience_counter = 0
    start_epoch = 0
    start_time = time.time()

    # ── Resume checkpoint ──
    resume_path = os.path.join(CFG.SEG_BENCH_DIR, "sam2_training_ckpt.pt")
    if os.path.exists(resume_path):
        ckpt = torch.load(resume_path, map_location=CFG.DEVICE, weights_only=False)
        current_sd = sam2_model.state_dict()
        current_sd.update(ckpt["model_decoder_state"])
        sam2_model.load_state_dict(current_sd)
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_dice = ckpt["best_dice"]
        best_state = ckpt["best_state"]
        patience_counter = ckpt["patience_counter"]
        print(f"  ♻️  RESUMING from epoch {start_epoch}, best_dice={best_dice:.4f}")

    # ── Training loop ──
    for epoch in range(start_epoch, CFG.MAX_EPOCHS):
        sam2_model.train()
        sam2_model.image_encoder.eval()
        train_loss = 0.0

        for img_list, masks, bboxes in s2_train_loader:
            masks  = masks.to(CFG.DEVICE)
            bboxes = bboxes.to(CFG.DEVICE)

            optimizer.zero_grad()

            # Encode image (frozen encoder, no grad)
            with torch.no_grad():
                predictor.set_image(img_list[0])
                features = predictor._features

            # Decode with grad
            pred = decode_with_box(sam2_model, features, bboxes[0])
            loss = criterion(pred, masks)

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(s2_train_ds)
        scheduler.step()

        # ── Validate ──
        sam2_model.eval()
        val_dices = []
        with torch.no_grad():
            for img_list, masks, bboxes in s2_val_loader:
                masks  = masks.to(CFG.DEVICE)
                bboxes = bboxes.to(CFG.DEVICE)

                predictor.set_image(img_list[0])
                features = predictor._features

                pred = decode_with_box(sam2_model, features, bboxes[0])
                prob = torch.sigmoid(pred.squeeze(0))
                p = (prob > 0.5).float()
                m = masks[0]
                if m.sum() == 0 and p.sum() == 0:
                    val_dices.append(1.0)
                elif m.sum() == 0 or p.sum() == 0:
                    val_dices.append(0.0)
                else:
                    inter = (p * m).sum()
                    dice = (2. * inter) / (p.sum() + m.sum())
                    val_dices.append(dice.item())

        mean_dice = np.mean(val_dices)
        median_dice = np.median(val_dices)

        if (epoch + 1) in [1, 5, 10, 15, 20, 30, 40, 50]:
            print(f"    Ep {epoch+1:3d}: loss={train_loss:.4f} "
                  f"mean_dice={mean_dice:.4f} median={median_dice:.4f}")

        if mean_dice > best_dice:
            best_dice = mean_dice
            best_state = {k: v.cpu().clone() for k, v in sam2_model.state_dict().items()
                          if "sam_mask_decoder" in k or "sam_prompt_encoder" in k}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= CFG.PATIENCE:
                print(f"    ⏹ Early stop at epoch {epoch + 1}")
                break

        if (epoch + 1) % 5 == 0:
            decoder_state = {k: v.cpu().clone() for k, v in sam2_model.state_dict().items()
                             if "sam_mask_decoder" in k or "sam_prompt_encoder" in k}
            torch.save({
                "epoch": epoch,
                "model_decoder_state": decoder_state,
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "best_dice": best_dice,
                "best_state": best_state,
                "patience_counter": patience_counter,
            }, resume_path)
            print(f"    💾 Mid-training checkpoint saved (epoch {epoch+1})")

    total_time = (time.time() - start_time) / 60

    # ── Final eval ──
    current_state = sam2_model.state_dict()
    current_state.update(best_state)
    sam2_model.load_state_dict(current_state)
    sam2_model.eval()

    final_dices = []
    with torch.no_grad():
        for img_list, masks, bboxes in s2_val_loader:
            masks  = masks.to(CFG.DEVICE)
            bboxes = bboxes.to(CFG.DEVICE)

            predictor.set_image(img_list[0])
            features = predictor._features

            pred = decode_with_box(sam2_model, features, bboxes[0])
            prob = torch.sigmoid(pred.squeeze(0))
            p = (prob > 0.5).float()
            m = masks[0]
            if m.sum() == 0 and p.sum() == 0:
                final_dices.append(1.0)
            elif m.sum() == 0 or p.sum() == 0:
                final_dices.append(0.0)
            else:
                inter = (p * m).sum()
                dice = (2. * inter) / (p.sum() + m.sum())
                final_dices.append(dice.item())

    dice_mean = np.mean(final_dices)
    dice_median = np.median(final_dices)
    dice_std = np.std(final_dices)

    print(f"\n  {'─'*60}")
    print(f"  📊 SAM2.1 (Hiera-S) — FINAL RESULTS")
    print(f"  {'─'*60}")
    print(f"  Mean Dice:    {dice_mean:.4f} ± {dice_std:.4f}")
    print(f"  Median Dice:  {dice_median:.4f}")
    print(f"  Total time:   {total_time:.1f} min")
    print(f"  Params:       {total_params/1e6:.1f}M total, {trainable/1e6:.2f}M trainable")
    print(f"\n  Comparison:")
    print(f"    U-Net (Eff-B3): {unet_results['dice_mean']:.4f} mean / {unet_results['dice_median']:.4f} median")
    print(f"    MedSAM (ViT-B): {medsam_results['dice_mean']:.4f} mean / {medsam_results['dice_median']:.4f} median")

    sam2_results = {
        "dice_mean": dice_mean,
        "dice_median": dice_median,
        "dice_std": dice_std,
        "dice_scores": final_dices,
        "total_time_min": total_time,
        "total_params": total_params,
        "trainable_params": trainable,
    }

    save_seg_checkpoint("sam2", sam2_results)

    decoder_path = os.path.join(CFG.SEG_BENCH_DIR, "sam2_decoder_best.pt")
    torch.save(best_state, decoder_path)
    print(f"  💾 Decoder weights: {decoder_path}")

    if os.path.exists(resume_path):
        os.remove(resume_path)
        print(f"  🧹 Cleaned mid-training checkpoint")

    del sam2_model, predictor, optimizer, scheduler
    torch.cuda.empty_cache()

print(f"\n✅ Cell 3 COMPLETE — SAM2 Mean Dice: {sam2_results['dice_mean']:.4f}")
print(f"✅ Ready for Cell 4: Aggregate & Pick Winner")


  🏋️  TRAINING: SAM2 on FUSeg
  📄 Ravi et al., Meta FAIR (2024)
  ✅ sam2 already installed
  ✅ Weights found: pretrained_weights/sam2.1_hiera_small.pt
  SAM2.1 (Hiera-S): 46.1M total
  Trainable (decoder+prompt): 11.74M
  Train: 610, Val: 400
  Image size: 1024×1024, Batch: 1 (per-image processing)
    Ep   1: loss=0.0652 mean_dice=0.9046 median=0.9291
    Ep   5: loss=0.0565 mean_dice=0.9043 median=0.9327
    💾 Mid-training checkpoint saved (epoch 5)
    Ep  10: loss=0.0512 mean_dice=0.9069 median=0.9317
    💾 Mid-training checkpoint saved (epoch 10)
    Ep  15: loss=0.0468 mean_dice=0.9126 median=0.9354
    💾 Mid-training checkpoint saved (epoch 15)
    Ep  20: loss=0.0423 mean_dice=0.9135 median=0.9356
    💾 Mid-training checkpoint saved (epoch 20)
    💾 Mid-training checkpoint saved (epoch 25)
    ⏹ Early stop at epoch 29

  ────────────────────────────────────────────────────────────
  📊 SAM2.1 (Hiera-S) — FINAL RESULTS
  ──────────────────────────────────────────────────────────

In [7]:
"""
═══════════════════════════════════════════════════════════════════════════════════
  Cell 4: AGGREGATE & PICK WINNER
═══════════════════════════════════════════════════════════════════════════════════
"""

print(f"\n{'='*80}")
print(f"  📊 SEGMENTATION BENCHMARK — FINAL COMPARISON")
print(f"{'='*80}")

# ── Load all results ──
unet_r = unet_results
medsam_r = medsam_results
sam2_r = sam2_results

rows = [
    {"Model": "U-Net (Eff-B3)", "Mean Dice": unet_r["dice_mean"],
     "Median Dice": unet_r["dice_median"], "Std": unet_r.get("dice_std", 0),
     "Params (M)": 13.2, "Time (min)": unet_r.get("total_time_min", 0),
     "Paper": "FUSegNet, Dhar et al. (2024)"},
    {"Model": "MedSAM (ViT-B)", "Mean Dice": medsam_r["dice_mean"],
     "Median Dice": medsam_r["dice_median"], "Std": medsam_r["dice_std"],
     "Params (M)": 93.7, "Time (min)": medsam_r["total_time_min"],
     "Paper": "Ma et al., Nat. Comm. (2024)"},
    {"Model": "SAM2.1 (Hiera-S)", "Mean Dice": sam2_r["dice_mean"],
     "Median Dice": sam2_r["dice_median"], "Std": sam2_r["dice_std"],
     "Params (M)": 46.1, "Time (min)": sam2_r["total_time_min"],
     "Paper": "Ravi et al., Meta FAIR (2024)"},
]

df = pd.DataFrame(rows).sort_values("Mean Dice", ascending=False)

print(f"\n  {'Model':<22s} {'Mean Dice':>12s} {'Median Dice':>14s} {'Std':>8s} {'Params':>10s} {'Time':>10s}")
print(f"  {'─'*78}")
for _, row in df.iterrows():
    winner = " 🏆" if row["Mean Dice"] == df["Mean Dice"].max() else ""
    print(f"  {row['Model']:<22s} {row['Mean Dice']:>12.4f} {row['Median Dice']:>14.4f} "
          f"{row['Std']:>8.4f} {row['Params (M)']:>8.1f}M {row['Time (min)']:>8.1f}m{winner}")

# ── Pick winner ──
winner_row = df.iloc[0]
winner_name = winner_row["Model"]
winner_dice = winner_row["Mean Dice"]

print(f"\n  {'='*78}")
print(f"  🏆 WINNER: {winner_name} — Mean Dice {winner_dice:.4f}")
print(f"  {'='*78}")

# ── Bar chart ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

models = df["Model"].tolist()
mean_dices = df["Mean Dice"].tolist()
median_dices = df["Median Dice"].tolist()
colors = ["#e74c3c" if d == max(mean_dices) else "#3498db" for d in mean_dices]

# Mean Dice
ax = axes[0]
bars = ax.bar(range(len(models)), mean_dices, color=colors, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, mean_dices):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.2%}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=15, ha="right", fontsize=10)
ax.set_ylabel("Mean Dice Score", fontsize=12)
ax.set_title("Segmentation Benchmark — Mean Dice", fontsize=13, fontweight="bold")
ax.set_ylim(0.75, 0.97)
ax.grid(axis="y", alpha=0.3)

# Median Dice
ax = axes[1]
bars = ax.bar(range(len(models)), median_dices, color=colors, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, median_dices):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.2%}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=15, ha="right", fontsize=10)
ax.set_ylabel("Median Dice Score", fontsize=12)
ax.set_title("Segmentation Benchmark — Median Dice", fontsize=13, fontweight="bold")
ax.set_ylim(0.85, 0.97)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(CFG.FIGURES_DIR, "seg_benchmark_comparison.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"\n  📈 Figure saved: {fig_path}")

# ── Save master results ──
master = {
    "benchmark_df": df.to_dict(),
    "winner": winner_name,
    "winner_dice": winner_dice,
    "unet": unet_r,
    "medsam": medsam_r,
    "sam2": sam2_r,
}
master_path = os.path.join(CFG.SEG_BENCH_DIR, "seg_benchmark_master.pt")
torch.save(master, master_path)
print(f"  💾 Master: {master_path}")

print(f"\n{'='*80}")
print(f"  ✅ SEGMENTATION BENCHMARK COMPLETE")
print(f"{'='*80}")
print(f"""
  RESULTS:
    U-Net (Eff-B3):    {unet_r['dice_mean']:.2%} mean / {unet_r['dice_median']:.2%} median  (13.2M params)
    MedSAM (ViT-B):    {medsam_r['dice_mean']:.2%} mean / {medsam_r['dice_median']:.2%} median  (93.7M, 4.1M trainable)
    SAM2.1 (Hiera-S):  {sam2_r['dice_mean']:.2%} mean / {sam2_r['dice_median']:.2%} median  (46.1M, 11.7M trainable)

  🏆 WINNER: {winner_name} at {winner_dice:.2%} mean Dice

  DECISION FOR WILLIE:
    • Segmentation backbone → {winner_name}
    • Detection baseline → RT-DETR (87.95% mAP@50) + YOLO (comparison)
    • Next: Rebuild MINI / BASE / XL with these components
""")


  📊 SEGMENTATION BENCHMARK — FINAL COMPARISON

  Model                     Mean Dice    Median Dice      Std     Params       Time
  ──────────────────────────────────────────────────────────────────────────────
  SAM2.1 (Hiera-S)             0.9174         0.9376   0.0869     46.1M     53.6m 🏆
  MedSAM (ViT-B)               0.9093         0.9333   0.0803     93.7M     36.6m
  U-Net (Eff-B3)               0.8459         0.9120   0.2021     13.2M     15.5m

  🏆 WINNER: SAM2.1 (Hiera-S) — Mean Dice 0.9174

  📈 Figure saved: artifacts/willie_v2/figures/seg_benchmark_comparison.png
  💾 Master: artifacts/willie_v2/seg_benchmark/seg_benchmark_master.pt

  ✅ SEGMENTATION BENCHMARK COMPLETE

  RESULTS:
    U-Net (Eff-B3):    84.59% mean / 91.20% median  (13.2M params)
    MedSAM (ViT-B):    90.93% mean / 93.33% median  (93.7M, 4.1M trainable)
    SAM2.1 (Hiera-S):  91.74% mean / 93.76% median  (46.1M, 11.7M trainable)

  🏆 WINNER: SAM2.1 (Hiera-S) at 91.74% mean Dice

  DECISION FOR WILLIE:
 

In [8]:
"""
═══════════════════════════════════════════════════════════════════════════════════
  Cell 5: PULL DETECTION RESULTS (RT-DETR + YOLO)
═══════════════════════════════════════════════════════════════════════════════════
"""

print(f"\n{'='*80}")
print(f"  📊 DETECTION BENCHMARK — RT-DETR vs YOLO")
print(f"{'='*80}")

DET_ROOT = os.path.join(CFG.ROOT, "artifacts", "willie_v2", "det_runs")

# ── RT-DETR results ──
rtdetr_csv = os.path.join(DET_ROOT, "rtdetr_l_willie", "results.csv")
rtdetr_results = None
if os.path.exists(rtdetr_csv):
    df_rt = pd.read_csv(rtdetr_csv)
    df_rt.columns = [c.strip() for c in df_rt.columns]
    best_idx = df_rt["metrics/mAP50(B)"].idxmax()
    rtdetr_results = {
        "model": "RT-DETR-L",
        "best_epoch": int(best_idx + 1),
        "mAP50": float(df_rt.loc[best_idx, "metrics/mAP50(B)"]),
        "mAP50_95": float(df_rt.loc[best_idx, "metrics/mAP50-95(B)"]),
        "precision": float(df_rt.loc[best_idx, "metrics/precision(B)"]),
        "recall": float(df_rt.loc[best_idx, "metrics/recall(B)"]),
    }
    print(f"\n  ✅ RT-DETR-L (from {rtdetr_csv})")
    print(f"     Best epoch:  {rtdetr_results['best_epoch']}")
    print(f"     mAP@50:      {rtdetr_results['mAP50']:.4f}")
    print(f"     mAP@50-95:   {rtdetr_results['mAP50_95']:.4f}")
    print(f"     Precision:   {rtdetr_results['precision']:.4f}")
    print(f"     Recall:      {rtdetr_results['recall']:.4f}")
else:
    print(f"\n  ❌ RT-DETR results.csv not found at: {rtdetr_csv}")

# ── YOLO results — search for it ──
yolo_results = None
yolo_candidates = [
    os.path.join(DET_ROOT, "yolov8m_willie", "results.csv"),
    os.path.join(DET_ROOT, "yolo8m_willie", "results.csv"),
    os.path.join(DET_ROOT, "yolov8_willie", "results.csv"),
]

# Also search dynamically
if os.path.isdir(DET_ROOT):
    for d in os.listdir(DET_ROOT):
        if "yolo" in d.lower():
            candidate = os.path.join(DET_ROOT, d, "results.csv")
            if candidate not in yolo_candidates:
                yolo_candidates.append(candidate)

print(f"\n  🔍 Searching for YOLO results...")
yolo_csv = None
for path in yolo_candidates:
    if os.path.exists(path):
        yolo_csv = path
        break
    else:
        print(f"     ✗ {path}")

if yolo_csv:
    df_yolo = pd.read_csv(yolo_csv)
    df_yolo.columns = [c.strip() for c in df_yolo.columns]
    best_idx = df_yolo["metrics/mAP50(B)"].idxmax()
    yolo_results = {
        "model": "YOLOv8m",
        "best_epoch": int(best_idx + 1),
        "mAP50": float(df_yolo.loc[best_idx, "metrics/mAP50(B)"]),
        "mAP50_95": float(df_yolo.loc[best_idx, "metrics/mAP50-95(B)"]),
        "precision": float(df_yolo.loc[best_idx, "metrics/precision(B)"]),
        "recall": float(df_yolo.loc[best_idx, "metrics/recall(B)"]),
    }
    print(f"\n  ✅ YOLOv8m (from {yolo_csv})")
    print(f"     Best epoch:  {yolo_results['best_epoch']}")
    print(f"     mAP@50:      {yolo_results['mAP50']:.4f}")
    print(f"     mAP@50-95:   {yolo_results['mAP50_95']:.4f}")
    print(f"     Precision:   {yolo_results['precision']:.4f}")
    print(f"     Recall:      {yolo_results['recall']:.4f}")
else:
    print(f"\n  ❌ YOLO results not found. Listing det_runs contents:")
    if os.path.isdir(DET_ROOT):
        for item in sorted(os.listdir(DET_ROOT)):
            full = os.path.join(DET_ROOT, item)
            if os.path.isdir(full):
                contents = os.listdir(full)
                print(f"     📁 {item}/ → {contents[:5]}...")
            else:
                print(f"     📄 {item}")
    else:
        print(f"     ❌ {DET_ROOT} does not exist")

# ── Head-to-Head Comparison ──
if rtdetr_results and yolo_results:
    print(f"\n  {'='*70}")
    print(f"  🏆 HEAD-TO-HEAD: RT-DETR-L vs YOLOv8m")
    print(f"  {'='*70}")
    print(f"  {'Metric':<16s} {'RT-DETR-L':>12s} {'YOLOv8m':>12s} {'Winner':>12s}")
    print(f"  {'─'*54}")

    metrics = ["mAP50", "mAP50_95", "precision", "recall"]
    labels  = ["mAP@50", "mAP@50-95", "Precision", "Recall"]

    rt_wins = 0
    yolo_wins = 0
    for metric, label in zip(metrics, labels):
        rt_val = rtdetr_results[metric]
        yo_val = yolo_results[metric]
        if rt_val > yo_val:
            winner = "RT-DETR ✅"
            rt_wins += 1
        elif yo_val > rt_val:
            winner = "YOLOv8m ✅"
            yolo_wins += 1
        else:
            winner = "TIE"
        print(f"  {label:<16s} {rt_val:>12.4f} {yo_val:>12.4f} {winner:>12s}")

    overall = "RT-DETR-L" if rt_wins > yolo_wins else "YOLOv8m" if yolo_wins > rt_wins else "TIE"
    print(f"  {'─'*54}")
    print(f"  Overall: {overall} ({rt_wins}-{yolo_wins})")

    # Save detection benchmark
    det_bench = {
        "rtdetr": rtdetr_results,
        "yolo": yolo_results,
        "winner": overall,
    }
    det_path = os.path.join(CFG.SEG_BENCH_DIR, "det_benchmark.pt")
    torch.save(det_bench, det_path)
    print(f"\n  💾 Detection benchmark saved: {det_path}")

elif rtdetr_results:
    print(f"\n  ⚠️  Only RT-DETR results available. YOLO comparison pending.")

print(f"\n{'='*80}")
print(f"  ✅ DETECTION NUMBERS PULLED")
print(f"{'='*80}")


  📊 DETECTION BENCHMARK — RT-DETR vs YOLO

  ✅ RT-DETR-L (from artifacts/willie_v2/det_runs/rtdetr_l_willie/results.csv)
     Best epoch:  65
     mAP@50:      0.8795
     mAP@50-95:   0.5738
     Precision:   0.8815
     Recall:      0.8938

  🔍 Searching for YOLO results...

  ✅ YOLOv8m (from artifacts/willie_v2/det_runs/yolov8m_willie/results.csv)
     Best epoch:  69
     mAP@50:      0.9122
     mAP@50-95:   0.5956
     Precision:   0.8745
     Recall:      0.8847

  🏆 HEAD-TO-HEAD: RT-DETR-L vs YOLOv8m
  Metric              RT-DETR-L      YOLOv8m       Winner
  ──────────────────────────────────────────────────────
  mAP@50                 0.8795       0.9122    YOLOv8m ✅
  mAP@50-95              0.5738       0.5956    YOLOv8m ✅
  Precision              0.8815       0.8745    RT-DETR ✅
  Recall                 0.8938       0.8847    RT-DETR ✅
  ──────────────────────────────────────────────────────
  Overall: TIE (2-2)

  💾 Detection benchmark saved: artifacts/willie_v2/seg_benc